#**Healthcare Insurance Cost Analysis Project:**

This project set out to explore how Data Analytics and AI tools can be use to extract inside information from raw datasets into meaningful insights that utilise for prediction and decesion manking in data driven organisation and for business success. The healthcare Insurance Cost Analysis Project seeks empirically to explore systematic approach to analyse raw datasets and its rationale to identify how different age, gender, habit and health condition affect health insureance cost in various location. In this Jupyter notebook, therefore, performing the steps: Section 1.0 to 1.5 data cleaning and Section 2.0 to 2.  data visualisation. 

*  For this analysis fetch data from Kaggle and save to local drive. Data then clean and processed for visualisation
*  Python programming language and its libraries are used in this analysis. Pyhton version use 3.12.8 and Libraries use - Numpy, Pandas, Matplotlib, Seaborn, plotly.

---

*Setting up working environment and directory*

Access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

Make the parent of the current directory the new current directory with os.path.dirname() to get the parent directory and os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Section 1

**Import, Clean, Transform and save as a clean dataset for visualisation and further analysis**

**1.0 Import Python libraries and Dataset**

Import pandas library which run on top of numerical python: Numpy backed by core python terminal. To import data file from souce can be performed in various ways by defining the file path on pandas data frame. Since, this project data already downloaded from Kaggle.com and store in local computer, local file path will use. However, direct source link can be used if required by providing full file path. For example- Local path: C:\\Users\\Documents\\insurance.csv
Or source path: https://www.kaggle.com/datasets/willianoliveiragibin/healthcare-insurance?select=insurance.csv

In [ ]:
import pandas as pd
import numpy as np

Import Data to pandas data frame and get an initial overview of data summary to generate thought process. Visualise some rows and columns. Below code will visualise 20 rows from source file.

In [ ]:
insurance_raw = pd.read_csv("C:\\Users\\skmra\\Documents\\healthcare_insurance_cost_analysis\\insurance.csv")

print(insurance_raw.shape)   # (891, 8)
insurance_raw.head(20)

**1.1 Check Summary of initial Dataset and find missing values**

To understand the dataset initially few steps are considers. First, check what type of data in the cell, any missing data or any empty cells, verify this by checking with non empty cell to match with with total cell. Also check any unque values that is not very common in the datasets. 

In [ ]:
before_summary = pd.DataFrame({
    "dtype": insurance_raw.dtypes,
    "missing_values": insurance_raw.isnull().sum(),
    "non_missing": insurance_raw.notnull().sum(),
    "unique_values": insurance_raw.nunique()
})
before_summary

After checking the datasets no missing values are found. However, if missing values are found then pandas missing values handling techniques will use.

**1.2 Check for duplicate data, Rename all the columns and create new columns**

Make a copy of the DataFrame and check for duplicate if found it will delete from the data frame by using drop_duplicates(inplace=True)

In [ ]:
insurance = insurance_raw.copy()
insurance.drop_duplicates(inplace=True)


Rename all the columns names to title case by using python for loop statesment. Change Bmi columns to B_M_I

In [ ]:
insurance.columns = [col.title() for col in insurance.columns]
insurance.rename(columns={'Bmi': 'B_M_I'}, inplace=True)
insurance.info()

view the unique values in each of the columns that will be categories:

In [ ]:
categorical_columns = ['Sex', 'Smoker', 'Region']

for col in categorical_columns:
    print(f"Unique values in {col} are : {insurance[col].unique()}")

And arrange these category data type and view the updated column types:

In [ ]:
categorical_columns = ['Sex', 'Smoker', 'Region']

for col in categorical_columns:
    insurance[col] = insurance[col].astype('category')
insurance.info()

**1.3 In this section each columns data will be group and analyse.** 

1.3.0 Add new Columns for age group 

In [ ]:
insurance['Age_Group'] = pd.cut(
    insurance['Age'],
    bins=[17, 26, 35, 45, 55, 100],
    labels=['Ages 18-25', 'Ages 26-34', 'Ages 35-44', 'Ages 45-54', 'Ages 55 and over'],
    right=False
)

insurance.head()

**1.3.1 Add new Columns for B_M_I group**

After researching online and publicly available information Age and B_M_I columns are group in the following ways. 
 age groups as age range as follows-

- `< 26` - 'Ages 18-25'
- `>= 26 < 35` - 'Ages 26-34'
- `>= 35 < 45` - 'Ages 35-44'
- `>= 44 < 55` - 'Ages 44-54'
- `>= 55` - 'Ages 55 and over'

and B_M_I group as follows

- `below 18.5` – you're in the underweight range
- `18.5 to 24.9` – you're in the healthy weight range
- `25 to 29.9` – you're in the overweight range
- `30 to 39.9` – you're in the obese range
- `40 or above` – you're in the severely obese range

Therefore, new group BMI_Group will be as follows-

- `< 18.5` - 'Underweight'
- `>= 18.5 < 25` - 'Healthy Weight'
- `>= 25 < 30` - 'Overweight'
- `>= 30 < 40` - 'Obese'
- `>= 40` - 'Severely Obese'

In [ ]:
insurance['BMI_Group'] = pd.cut(
    insurance['B_M_I'],
    bins=[0, 18.5, 25, 30, 40, 100],
    labels=['Underweight', 'Healthy Weight', 'Overweight', 'Obese', 'Severely Obese'],
    right=False
)

insurance.head()

---

**1.3.2 Add new Columns for insurance cost plan**

For Futher analysis in the dataset, will add Plan column which will create based on the individual with children or no children. If individual have no children the plan will be standard. If individual have children then it will be a Family plan.

In [ ]:
insurance['Plan'] = np.where(insurance['Children'] > 0, 'Family', 'Standard')
insurance['Plan'] = insurance['Plan'].astype('category')
insurance.head()

**1.3.3 Fromating the title case all the values have a capital letter first**

In [ ]:
insurance['Sex'] = insurance['Sex'].cat.rename_categories(lambda x: x.title())
insurance['Smoker'] = insurance['Smoker'].cat.rename_categories(lambda x: x.title())
insurance['Region'] = insurance['Region'].cat.rename_categories(lambda x: x.title())
insurance.head()

**1.3.4 Add a new columns to calculate per person cost**

By considering individuals and number of children on the insurance plan

In [ ]:
insurance['Charges_Per_Person'] = insurance['Charges'] / (insurance['Children'] + 1)
insurance.head()

**1.3.4 Add outlier columns and numeric columns for categories** 


In [ ]:
def check_outlier(s):
    """ Calculating inter quartile range and returning True or False"""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (s < low) | (s > high)

insurance['Charges_Outlier'] = check_outlier(insurance['Charges'])
insurance['Charges_Per_Person_Outlier'] = check_outlier(insurance['Charges_Per_Person'])
insurance.head()

In [ ]:
cat_cols = ['Sex', 'Smoker', 'Region', 'Plan', 'BMI_Group', 'Age_Group']
for col in cat_cols:
    insurance[f"{col}_num"] = insurance[col].cat.codes

insurance.head()

**1.3.5 Rearranging the columns to improve readability**

In [ ]:
insurance = insurance[[
    'Age', 'Age_Group', 'Age_Group_num',
    'Sex', 'Sex_num',
    'B_M_I', 'BMI_Group', 'BMI_Group_num',
    'Children', 'Plan', 'Plan_num',
    'Smoker', 'Smoker_num',
    'Region', 'Region_num',
    'Charges', 'Charges_Per_Person',
    'Charges_Outlier', 'Charges_Per_Person_Outlier'
]]
insurance.head()

**1.4 Comparing raw data set with newly cleaned dataset**

**1.4.0 Create after summary of the data set**

In [ ]:
after_summary = pd.DataFrame({
    "dtype": insurance.dtypes,
    "missing_values": insurance.isnull().sum(),
    "non_missing": insurance.notnull().sum(),
    "unique_values": insurance.nunique()
})
after_summary

**1.4.0 Create before summary of the data set for a view**

In [ ]:
before_summary

** 1.5 And finally the clean dataset to will save as a clean file**

In [ ]:
insurance.to_csv(r"C:\Users\skmra\Documents\healthcare_insurance_cost_analysis\insurance_cleaned.csv", index=False)

# Section 2

Clean Data analysis and Visualisation 

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
